# 05 · Which traces make a model imprecise?

LMP is additive over traces. Every unseen trace the model accepts owns an exact share of the imprecision:

$$1 - \mathrm{LMP}_\mu(L, M) = \sum_{\sigma \in \mathcal L(M)\setminus\tilde L} \frac{\mu(\sigma)}{\mu(\mathcal L(M))}$$

`imprecision_attribution` lists the heaviest such traces. It runs a best-first search over the DFA, so it works for infinite languages too: traces come out in non-increasing mass order.

In [1]:
import os, warnings
os.environ["PM4PY_SHOW_PROGRESS_BAR"] = "False"
warnings.filterwarnings("ignore", category=DeprecationWarning)

import language_mass_precision as lmp
import pm4py
import pandas as pd
pd.set_option("display.precision", 6)

In [2]:
log = [("register", "check", "approve", "notify"),
       ("register", "check", "reject", "notify"),
       ("register", "check", "check", "approve", "notify")]
model = pm4py.parse_process_tree(
    "->('register', *('check', tau), X('approve', 'reject', ->('escalate', 'approve')), X('notify', tau))")

result = lmp.precision_language_mass(log, model)
print(result, "\nimprecision =", result.imprecision)
table = lmp.imprecision_attribution(log, model, top_k=12)
table.assign(trace=table["trace"].map(" → ".join))[["rank", "trace", "length", "imprecision_share", "cumulative_share"]]

LMPResult(status=exact, value=0.108323, certification='full_language_sparse_solve') 
imprecision = 0.8916766796832638


,rank,trace,length,imprecision_share,cumulative_share
0,1,register → check → approve,3,0.348043,0.348043
1,2,register → check → reject,3,0.348043,0.696087
2,3,register → check → check → approve,4,0.050498,0.746585
3,4,register → check → check → reject,4,0.050498,0.797083
4,5,register → check → escalate → approve,4,0.050498,0.847581
5,6,register → check → check → check → approve,5,0.007327,0.854908
6,7,register → check → check → check → reject,5,0.007327,0.862235
7,8,register → check → check → escalate → approve,5,0.007327,0.869562
8,9,register → check → check → reject → notify,5,0.007327,0.876889
9,10,register → check → escalate → approve → notify,5,0.007327,0.884216


The top rows point to concrete over-generalizations: the optional `notify` (the silent branch lets a case end without it) and the escalation path. These are things a modeller can act on.

The estimator exposes the same table. Here we compare how two priors distribute blame.

In [3]:
for prior in ("geometric", "power_law"):
    estimator = lmp.LanguageMassPrecision(length_prior=prior).fit(log)
    table = estimator.attribution(model, top_k=5)
    print(f"--- {prior}: LMP in [{estimator.score(model).lower:.4f}, {estimator.score(model).upper:.4f}]")
    print(table.assign(trace=table["trace"].map(" → ".join))[["trace", "share_lower", "share_upper"]].to_string(index=False))

--- geometric: LMP in [0.1083, 0.1083]
                                trace  share_lower  share_upper
           register → check → approve     0.348043     0.348043
            register → check → reject     0.348043     0.348043
   register → check → check → approve     0.050498     0.050498
    register → check → check → reject     0.050498     0.050498
register → check → escalate → approve     0.050498     0.050498
--- power_law: LMP in [0.0970, 0.0970]
                                trace  share_lower  share_upper
           register → check → approve     0.365477     0.365477
            register → check → reject     0.365477     0.365477
   register → check → check → approve     0.045502     0.045502
    register → check → check → reject     0.045502     0.045502
register → check → escalate → approve     0.045502     0.045502


Under the power-law prior the denominator is a certified interval, so each share is reported as `[share_lower, share_upper]`.